<a href="https://colab.research.google.com/github/SkellXC/Game-Store-Management-Software/blob/main/mainUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Database & Utility Functions


**Author:** F535453

**Date:** 12/12/2025

 **Description:** This module serves as the Data Access Layer for the
    Game Store Management System. It contains utility functions to read
    text-based databases (CSV format), calculate store statistics (occupancy,
    inventory), and validate business logic rules (such as rental limits and
    membership status).

**Usage:**
Run this cell first to initialize the constants and helper functions required
by the GUI cells.

In [103]:
"""
Cell: setupData
Description: Generates all required helper functions for database interaction.
"""
import subscriptionManager
import feedbackManager
import csv
# --- CONSTANTS ---
rentalLimits = {
    "Standard": 1,
    "Premium": 3
}

rentalFile = "Rental.txt"
subscriptionFile = "Subscription_Info.txt"

def _readCsv(filename):
    """
    Reads a CSV-style text file and returns a list of dictionaries.

    Parameters:
        filename (str): The name of the file to read.

    Returns:
        list: A list of dictionaries representing the rows.
    """
    data = []
    try:
        with open(filename, 'r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                data.append(dict(row))
    except FileNotFoundError:
        return []
    return data

def getCustomerInfo(customerID):
    """
    Retrieves user details from the subscription database.

    Parameters:
        customerID (str): The unique ID of the customer.

    Returns:
        dict: The customer row if found, otherwise None.
    """
    allCustomers = _readCsv(subscriptionFile)
    for person in allCustomers:
        if person['CustomerID'] == customerID:
            return person
    return None

def checkPremium(customerID):
    """
    Checks if the customer has a valid Premium subscription.

    Parameters:
        customerID (str): The unique ID of the customer.

    Returns:
        bool: True if premium, False otherwise.
    """
    subscriptions = subscriptionManager.load_subscriptions()
    isSubbed = subscriptionManager.check_subscription(customerID, subscriptions)
    return isSubbed


def getActiveRentals(customerID):
    """
    Counts how many games a specific customer currently has rented.

    Parameters:
        customerID (str): The unique ID of the customer.

    Returns:
        int: The number of active rentals (rows with no return date).
    """
    allRentals = _readCsv(rentalFile)
    count = 0
    for row in allRentals:
        # Check if ID matches and the 'Return Date' column is empty
        if (row['Rental Customer ID'] == customerID) and (row['Rental Date'] == ""):
            count += 1
    return count

def checkRentalEligibility(customerID):
    """
    Determines if a user is allowed to rent more games based on their tier.

    Parameters:
        customerID (str): The unique ID of the customer.

    Returns:
        tuple: (bool, str) -> (Allowed?, Reason/Message)
    """
    user = getCustomerInfo(customerID)
    if user is None:
        return False, "Invalid Customer ID."

    tier = user["SubscriptionType"]
    # Default to 1 if tier is not found in limits
    limit = rentalLimits.get(tier, 1)
    currentCount = getActiveRentals(customerID)

    if currentCount >= limit:
        return False, f"Rental limit reached ({currentCount}/{limit})."

    return True, f"Eligible ({tier})"

def checkMembership(customerID):
    """
    Simple check to see if the ID exists in the system.

    Parameters:
        customerID (str): The unique ID of the customer.

    Returns:
        bool: True if user exists, False otherwise.
    """
    user = getCustomerInfo(customerID)
    return False if user is None else True


def inventoryCount():
    """
    Calculates the total inventory and the number of currently rented items.

    Returns:
        tuple: (totalGames, gamesRented)
    """
    total = 0
    rented = 0
    # Count board games
    try:
        with open("Board_Game_Info.txt","r") as f:
            total += sum(1 for row in csv.DictReader(f))
    except FileNotFoundError:
        pass
    # Count video games
    try:
        with open("Video_games_Info.txt","r") as f:
            total += sum(1 for row in csv.DictReader(f))
    except FileNotFoundError:
        pass

    # Count active rentals (no return date)
    try:
        with open("Rental.txt","r") as f:
            for row in csv.DictReader(f):
                if row["Return Date"].strip() == "":
                    rented += 1
    except FileNotFoundError:
        pass

    return total, rented


def getCurrentOccupancy():
    """
    Calculates how many people are booked for the current time slot today.

    Returns:
        int: Total number of people (Customers + Guests).
    """
    now = datetime.now()
    today = now.strftime("%Y-%m-%d")
    hour = now.hour

    # Determine the current time slot
    if 14 <= hour < 18:
        slot = ["2pm", "2pm-6pm"]
    elif 18 <= hour < 22:
        slot = ["6pm", "6pm-10pm"]
    else:
        # Store is closed outside these hours
        return 0

    occupancy = 0

    try:
        with open("Bookings.txt","r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Filter by todays date
                if row["Booking Date"] == today:
                    # Filter by the time slot
                    if row["Time"] in slot:
                        # Add the member plus their guests
                        guests = int(row(["Number of Guests"]))
                        occupancy += (1 + guests)
    except FileNotFoundError:
        pass
    return occupancy


def getStoreStatus():
    """
    Generates the live status text for the dashboard.

    Returns:
        tuple: (DateString, TimeString, OpenStatusHTML, Message)
    """
    now = datetime.now()
    today = now.strftime("%Y-%m-%d")
    hour = now.hour
    displayTime = now.strftime("%I %p")

    # Check store opening hours (2pm to 10pm)
    if 14 <= hour < 22:
        status = "<span style='color: green; font-weight: bold;'>OPEN</span>"
        msg = f"Happening Now: {displayTime} Session"
    else:
        status = "<span style='color: red; font-weight: bold;'>CLOSED</span>"
        msg = "Out of Hours (Open 2pm - 10pm)"

    return today, displayTime, status, msg

# Search System & Inventory Logic


 **Author:**  F535453

 **Date:** 20/12/2025

 **Description:** This cell implements the search functionality for the
    store's inventory. It allows the user to filter games by Type (Board/Video),
    Title, and Genre. It performs a cross-reference check against the 'Rental.txt'
    database to determine real-time availability (Green = Available, Red = Rented).

**Usage:**
1. Select a **Game Type** from the dropdown.
2. (Optional) Enter a partial **Title** (e.g., "zel" for "Zelda").
3. (Optional) Enter a partial **Genre** (e.g., "rpg").
4. Click **Search** to view the results table.


In [104]:
"""
Cell: searchTab
Description: Implements the search UI and logic for filtering the game inventory.
"""
from IPython.core.display import clear_output
from ipywidgets import widgets

def readData(filename):
    """
    Reads a CSV-style text file and returns a list of dictionaries.

    Parameters:
        filename (str): The path to the text file (e.g., 'Rental.txt').

    Returns:
        list: A list of dictionaries where keys match the CSV headers.
    """
    data = []
    try:
        with open(filename, 'r') as file:
            # csv.DictReader automatically uses the first row as keys (headers)
            reader = csv.DictReader(file)
            for row in reader:
                data.append(dict(row))
    except FileNotFoundError:
        return []
    return data

def tokenMatch(searchTerm,text):
    """
    Checks if a search term matches the start of any word in the text.
    Example: "zel" matches "The Legend of Zelda" (via "Zelda").

    Parameters:
        searchTerm (str): The user's input string.
        text (str): The full text to search against (e.g., Game Title).

    Returns:
        bool: True if a match is found, False otherwise.
    """
    if searchTerm == "":
        return True

    # Split text into individual words to check starts of words specifically
    words = text.lower().split()
    for word in words:
        if word.startswith(searchTerm):
            return True
    return False


def searchInventory(gameType,searchTitle,searchGenre):
    """
    Filters the inventory based on criteria and checks availability.

    Parameters:
        gameType (str): Category filter ("Board Game", "Video Game", "Any").
        searchTitle (str): The partial title to search for.
        searchGenre (str): The partial genre to search for.

    Returns:
        list: A list containing two lists [[Available Games], [Rented Games]].
    """

    genreOnly = True if searchTitle == "" else False
    titleOnly = True if searchGenre == "" else False

    # Normalize the Inputs
    searchTitle = searchTitle.lower()
    searchGenre = searchGenre.lower()

    # Load data
    rentals = readData("Rental.txt")

    if gameType == "Board Game":
        inventory = readData("Board_Game_Info.txt")
    elif gameType == "Video Game":
        inventory = readData("Video_Game_Info.txt")
    else:
        # Combine both lists for "Any"
        inventory = (readData("Board_Game_Info.txt") +
                             readData("Video_Game_Info.txt"))

    # Sort results alphabetically by Name
    inventory = sorted(inventory, key=lambda x: x.get('Name', '').lower())

    # Create a set of currently rented Game IDs for fast lookup
    # A game is rented if 'Return Date' is empty string ""
    rentedIDs = {r['Game ID'].lower() for r in rentals if r['Return Date'] == ""}

    results = []

    # Logic: Filter by Title AND Genre
    # Note: If a field is empty, tokenMatch returns True (ignoring that filter)
    if genreOnly:
        for item in inventory:
            gameGenre = str(item.get("Genre","")).lower()
            if tokenMatch(searchGenre, gameGenre):
                results.append(item)

  # Title Search (with optional Genre filter)
    else:
        # Search with Title (and optional Genre)
        for item in inventory:
            gameName = str(item.get("Name","")).lower()
            gameGenre = str(item.get("Genre","")).lower()

            # Check Title (Mandatory)
            nameMatches = tokenMatch(searchTitle,gameName)
            # Check Genre (Optional: matches if blank OR if found)
            genreMatches = (searchGenre == "") or tokenMatch(searchGenre,gameGenre)

            # Only add if both conditions are met
            if nameMatches and genreMatches:
                results.append(item)

    # Split into separate lists based on availability
    unavailableGames = []
    availableGames = []

    for game in results:
        resultGameIDs = str(game.get("Game ID","")).lower()
        if resultGameIDs in rentedIDs:
            unavailableGames.append(game)
        else:
            availableGames.append(game)

    # Return the separated lists for the UI to render in different colors
    return [availableGames,unavailableGames]

def onSearchClick(b):
    """
    Event handler for the Search button.
    Triggers the search logic and renders the HTML results table.

    Parameters:
        b (obj): The button instance that was clicked.
    """
    with searchOutput:
        clear_output()

        searchGameType = gameTypeWidget.value
        searchGameName = gameTitleInputBox.value.strip()
        searchGameGenre = gameGenreInputBox.value.strip()


        # Perform search
        availableGames, unavailableGames = searchInventory(searchGameType,
                                        searchGameName,searchGameGenre)

        if len(availableGames) == 0 and len(unavailableGames) == 0:
            display(widgets.HTML("<b>There are no games in stock that match these criteria.</b>"))
            return

        # Build HTML Table
        html = """
            <style>
                table { width: 100%; border-collapse: collapse; }
                th { text-align: left; background-color: #E2DFD2;color:#000000; padding: 8px; border-bottom: solid #ddd; }
                td { padding: 8px; border-bottom: 1px solid #ddd; }
            </style>
            <table>
                <tr>
                    <th>ID</th>
                    <th>Name</th>
                    <th>Genre</th>
                    <th>Status</th>
                </tr>
            """
        # Add the available Rows
        for game in availableGames:
            html += f"""
                <tr>
                    <td>{game.get('Game ID')}</td>
                    <td>{game.get('Name')}</td>
                    <td>{game.get('Genre')}</td>
                    <td style="color: green; font-weight: bold;">Available</td>
                </tr>
                """

        # Add the rented rows
        for game in unavailableGames:
            html += f"""
                <tr>
                    <td>{game.get('Game ID')}</td>
                    <td>{game.get('Name')}</td>
                    <td>{game.get('Genre')}</td>
                    <td style="color: red; font-weight: bold;">Rented</td>
                </tr>
                """
        html += "</table>"
        display(widgets.HTML(html))



# Session Booking System

 **Author:** F535453

 **Date:** 20/12/2025

 **Description:** This cell handles the booking logic for face-to-face
    sessions. It allows subscribers to book 4-hour slots (2pm-6pm or 6pm-10pm).
    It ensures that:
    1. Maximum store capacity (50 people).
    2. Guest limits (Premium = 3, Standard = 0).
    3. Duplicate booking prevention (one slot per person per day).

**Usage:**
1. Enter a valid **Customer ID**.
2. Select a **Date** using the picker.
3. Choose a **Time Slot** and **Number of Guests**.
4. Click **Book Session** to commit the reservation to 'Bookings.txt'.

In [105]:
"""
Cell: sessionTab
Description: Implements the UI and logic for booking store sessions.
"""
def createBooking(customerID,date,timeSlot,guests):
    """
    Validates and saves a new booking to the database.

    Parameters:
        customerID (str): The ID of the user booking the slot.
        date (obj): The datetime.date object from the date picker.
        timeSlot (str): The selected time (e.g., "2pm-6pm").
        guests (int): Number of guests accompanying the user.

    Returns:
        tuple: (bool, str) -> (Success?, Message)
    """
    maxCapacity = 50
    guests = int(guests)
    # 1. Validation: Membership & Tier Privileges
    isMember = checkMembership(customerID)
    isPremium = checkPremium(customerID)

    # Standard members cannot bring guests; Premium can bring up to 3
    if not isMember:
        return False, "Invalid ID or User is not a member"
    if guests > 1 and not isPremium:
        return False, "User can bring up to 1 guest under their subscription"

    # 2. Validation: Capacity & Duplicates
    currentBookingList = readData("Bookings.txt")
    peopleCount = 0
    dateString = str(date)

    for row in currentBookingList:
        # If the same person tries to book the same slot, return error message
        if (row['Customer ID'] == customerID and
            row['Booking Date'] == dateString and
            row['Time'] == timeSlot):
            return False, "User already has a booking for this slot."

        # Calculate Capacity
        if row['Booking Date'] == dateString and row['Time'] == timeSlot:
            bookedGuests = int(row['Number of Guests'])
            peopleCount += (1 + bookedGuests)

    if peopleCount + 1 > maxCapacity:
        return False, f"Store full. Can only accomodate {maxCapacity} max."

    # 3. Save Booking
    booking = f'\n"{customerID}","{date}","{timeSlot}","{guests}"'

    try:
        with open("Bookings.txt","a") as f:
            f.write(booking)
            return True, "Booking Successful!"
    except Exception as e:
        return False, f"File error: {e}"



def onBookClick(b):
    """
    Event handler for the Booking button.
    Validates inputs and calls the creation logic.
    """
    with bookingOutput:
        clear_output()

        customerID = customerIDInputBox.value.strip()
        bookingDate = dateInputBox.value
        bookingTime = sessionTimeWidget.value
        guestAmount = guestAmountWidget.value

        if not customerID or not bookingDate:
            noID = "Please enter a customer ID"
            noDate = "Please enter a valid date"
            bothEmpty = "Please enter a customer ID and Date"
            if not customerID and bookingDate:
                message = noID
            elif not bookingDate and customerID:
                message = noDate
            else:
                message = bothEmpty

            display(widgets.HTML(f"<b style='color: red;'>{message}</b>"))
            return

        success, msg = createBooking(customerID, bookingDate, bookingTime, guestAmount)

        if success:
            display(widgets.HTML(f"<b style='color: green;'>{msg}</b>"))
            customerIDInputBox.value = "" # Clear input on success
        else:
            display(widgets.HTML(f"<b style='color: red;'>{msg}</b>"))



# Rental Processing System


 **Author:** F535453

 **Date:** 2025-12-20

 **Description:** This cell implements the logic and UI for the Rental system.


1. **Availability Checking:** Verifying if a game is in stock and not
       currently rented out.

2. **Eligibility Checking:** Ensuring the user has a valid subscription and
       has not exceeded their rental limit.

3. **Transaction Processing:** Recording valid rentals to 'Rental.txt'.

4. **Status Dashboard:** Providing immediate visual feedback on user status
       and rental decisions.


**Usage:**
1. Enter the **Customer ID** and **Game ID**.
2. Click **Check Details** to validate eligibility and stock.
3. If eligible (Green Status), click **Confirm Rental** to finalize.

In [106]:
from datetime import date
"""
Cell: rentTab
Description: Implements the UI and logic for renting games to customers.
"""
def checkAvailability(gameID):
    """
    Checks if a specific game is currently available for rent.
    It verifies existence in the inventory and checks active rental status.

    Parameters:
        gameID (str): The unique ID of the game (e.g., 'bg01').

    Returns:
        tuple: (bool, str) -> (Is Available?, Reason)
    """
    rentalFile = _readCsv("Rental.txt")

    # Load complete inventory
    allGames=_readCsv("Video_Game_Info.txt")+ _readCsv("Board_Game_Info.txt")

    # Validate that the game ID exists
    gameExists = False
    for item in allGames:
        if gameID.lower() == item["Game ID"].lower():
            gameExists = True
            break

    if not gameExists:
        return False, "Invalid ID entered"

    # Check rental history for any active rentals
    for row in rentalFile:
        # Check if IDs match
        if row["Game ID"].strip().lower() == gameID.strip().lower():
            # Check if the return date is empty (Active rental)
            if row["Return Date"].strip() == "":
                return False, "Game is currently being rented"

    # If the loop finishes with no active rentals found, its available
    return True, "Game is in stock and available"


def processRental(customerID, gameID):
    """
    Finalizes the rental transaction by writing to the database.

    Parameters:
        customerID (str): The ID of the renting customer.
        gameID (str): The ID of the game being rented.

    Returns:
        tuple: (bool, str) -> (Success?, Message)
    """
    # Check if the customer is eligible to rent
    canUserRent, reason = checkRentalEligibility(customerID)
    if not canUserRent:
        return(reason)

    # Check if the game is available
    isAvailable, reason = checkAvailability(gameID)
    if not isAvailable:
        return(reason)

    currentDate = date.today().strftime("%Y-%m-%d")
    dbEntry = f'\n"{gameID}","{currentDate}","","{customerID}"'
    try:
        with open("Rental.txt", "a") as f:
            f.write(dbEntry)
        return True, "Rental Confirmed!"
    except Exception as e:
        return False, f"File Error: {e}"



# --- 2. HELPER FUNCTIONS ---

def getCurrentRentals(customerID):
    """
    Reads Rental.txt and returns a list of Game IDs currently held by the user.

    Parameters:
        customerID (str): The user to check.

    Returns:
        list: A list of Game IDs (strings).
    """
    currentGames = []
    targetID = customerID.strip().lower()

    try:
        with open("Rental.txt", "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Check ID and ensure Return Date is empty
                rentalID = row.get("Rental Customer ID", "").strip().lower()
                rentalDate = row.get("Return Date", "").strip()

                if rentalID == targetID and rentalDate == "":
                    currentGames.append(row.get("Game ID", "Unknown"))
    except FileNotFoundError:
        return []

    return currentGames

def updateCustomerDisplay(tier, isActive, currentRentalsList):
    """
    Updates the Customer Status Box HTML with subscription details.

    Parameters:
        tier (str): The subscription tier (e.g., 'Premium').
        isActive (bool): Subscription status.
        currentRentalsList (list): List of currently rented game IDs.
    """
    if isActive:
        colour, statusIcon  = "lightgreen", "Active"
    else:
        colour, statusIcon = "#FFCCCB", "Inactive"

    if len(currentRentalsList) > 0:
        rentals_str = ", ".join(currentRentalsList)
    else:
        rentals_str = "-"


    html = f'''
        <div style="background-color: {colour}; color: black; border: 1px solid #ccc;
        padding: 10px; height: 100px; overflow-y: auto;">
            <b>👤 Rental Status:</b> {statusIcon}<br>
            <hr style="margin: 5px 0;">
            <b>Tier:</b> {tier}<br>
            <b>Currently Renting:</b> {rentals_str}

        </div>
        '''
    customerStatusBox.value = html

def updateEligibilityDisplay(canRent, message):
    """
    Updates the 'Rental Decision' box and toggles the Rent Button state.

    Parameters:
        canRent (bool): Authorization status.
        message (str): Explanation for the decision.
    """
    bgColour = "lightgreen" if canRent else "#FFCCCB"
    rentGameButton.disabled = not canRent
    # Enables the button only if it passes the checks
    html = f"""
        <div style="
            background-color: {bgColour};
            border: 1px solid #999;
            padding: 10px;
            min-height: 100px; /* changed from height to min-height */
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            text-align: center;
        ">
            <b style="font-size: 14px; color: black; margin-bottom: 5px;">
                {'User is eligible to rent' if canRent else 'Item cannot be rented'}
            </b>
            <span style="font-size: 12px; color: black; line-height: 1.2;">
                {message}
            </span>
        </div>
    """
    rentEligibilityBox.value = html

def onCheckClick(b):
    """
    Event handler for 'Check Details' button.
    Orchestrates the validation logic and updates UI widgets.
    """
    with outputLog:
        outputLog.clear_output()
        try:
            customerID = rentCustomerIDInputBox.value.strip()
            gameID = gameIDInputBox.value.strip()

            canRent, msg = checkRentalEligibility(customerID)
            user = getCustomerInfo(customerID)
            tier = user["SubscriptionType"] if user else "Unknown User ID"

            currentGames = getCurrentRentals(customerID)
            updateCustomerDisplay(tier,canRent,currentGames)
            isGameAvailable, gMsg = checkAvailability(gameID)

            if not canRent:
                finalMessage = f"Customer Issue: {msg}"
            elif not isGameAvailable:
                finalMessage = f"Game Issue: {gMsg}"
            else:
                finalMessage = "All checks passed. Game can be rented"

            decision = canRent and isGameAvailable
            updateEligibilityDisplay(decision, finalMessage)

        except Exception as e:
            print(f"Found error{e}")



def onRentClick(b):
    """
    Event handler for 'Confirm Rental' button.
    Commits the rental to the database.
    """
    with outputLog:
        outputLog.clear_output()
        customerID = rentCustomerIDInputBox.value.strip()
        gameID = gameIDInputBox.value.strip()
        success, msg = processRental(customerID, gameID)

        if success:
            print(f"{msg}")
            # Optional: Reset inputs after success
            rentCustomerIDInputBox.value = ""
            gameIDInputBox.value = ""
            rentGameButton.disabled = True # Disable until checked again
        else:
            print(f"{msg}")



# Return & Feedback System

**Program Information:**
* **Author:** F535453
* **Date:** 2025-12-20
* **Description:** This cell implements the logic for returning games.
It performs two main functions:
1. **Inventory Management:** Updates 'Rental.txt' to mark a specific game
rental as 'returned' by adding today's date.
2. **Feedback Collection:** Captures user ratings (1-5) and comments,
formatting them into a structured string before saving them via the `feedbackManager` module.

**Usage:**
1. Enter the **Game ID** of the item being returned.
2. (Optional) Provide **Rating**, **Replayability**, **Condition**, and **Comments**.
3. If the item was unused, check "Item unused" to disable feedback.
4. Click **Accept Return** to process the transaction.

In [107]:

def updateReturnRecord(gameID):
    """
    Reads Rental.txt, finds the active rental for this game,
    updates the Return Date, and RE-WRITES the whole file.
    """
    gameID = gameID.strip().lower()
    currentDate = date.today().strftime("%Y-%m-%d")

    updatedRows = []
    foundActive = False
    try:
        # 1. Read all rows
        with open("Rental.txt", "r") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames

            for row in reader:
                # Check for match: Same Game ID AND Empty Return Date
                rowGameID = row.get("Game ID", "").strip().lower()
                returnDate = row.get("Return Date", "").strip()

                if rowGameID == gameID and returnDate == "":
                    # FOUND IT! Update the return date
                    row["Return Date"] = currentDate
                    foundActive = True

                updatedRows.append(row)
        if foundActive:
            with open("Rental.txt", "w", newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(updatedRows)
            return True, "Return Processed."
        else:
            return False, " No active rental found for this Game ID."

    except Exception as e:
        return False, f"File Error: {e}"


def on_return_click(b):
    with returnOutput:
        returnOutput.clear_output()

        # 1. Get Inputs
        gameID = returnGameID.value.strip()

        if not gameID:
            print("Please enter a Game ID.")
            return

        # 2. Process the Return (Database)
        success, msg = updateReturnRecord(gameID)

        if success:
            print(msg)
            # 3. Handle Feedback (If the user wants to leave it)
            # We check if the "No Feedback" box is UNCHECKED (False)
            if not noFeedbackCheckbox.value:
                try:
                    # A. Bundle the extra data into the string
                    condition = conditionDropdown.value
                    replay = replayRatingSlider.value
                    raw_comment = returnComment.value

                    # Format: "[Condition: Good | Replay: 3/5] The comment text"
                    bundled_comment = f"[Condition: {condition} | Replay: {replay}/5] {raw_comment}"

                    # B. Call the strict .pyc function
                    # It only accepts: (ID, Rating, Comment, Filename)
                    feedbackManager.add_feedback(
                        gameID,
                        int(generalRatingSlider.value),
                        bundled_comment,
                        "Game_Feedback.txt"
                    )
                    print("Feedback saved!")

                except Exception as e:
                    print(f"Error saving feedback: {e}")

            # 4. Reset the UI
            returnGameID.value = ""
            returnComment.value = ""
            noFeedbackCheckbox.value = False
            generalRatingSlider.value = 3
            replayRatingSlider.value = 3

        else:
            print(msg)

def toggleFeedback(change):
    should_disable = change['new']

    generalRatingSlider.disabled = should_disable
    replayRatingSlider.disabled = should_disable
    conditionDropdown.disabled = should_disable
    returnComment.disabled = should_disable



# Pruning

In [108]:

from datetime import datetime
import re
# Constants that the algorithms base pruning on
CURRENT_DATE = datetime(2025, 12, 1) # Pretend "Today's Date"
MIN_UNIQUE_CUSTOMERS = 5 # For Breadth
CHURN_RISK_RATING = 4.0  # If rating is > 4.0, it's a "Favorite"
SMART_SCORE_REQUIREMENT = 0.4

# --- 1. PARSER HELPER ---
def parseFeedbackData():
    """
    Reads the file and builds a profile for each game.
    """
    gameStats = {}

    try:
        with open("Rental.txt", "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                gameID = row["Game ID"].strip()
                customerID = row["Rental Customer ID"].strip()
                # Parsing Date (assuming format YYYY-MM-DD)
                try:
                    rentalDate = datetime.strptime(row["Rental Date"], "%Y-%m-%d")
                except ValueError:# Skip invalid
                    continue

                if gameID not in gameStats:
                    gameStats[gameID] = {
                        "ratings": [],
                        "dates": [],
                        "customers": set(), # Use a set to count unique people automatically
                        "replayability": [],
                        "lastCondition":"Unknown"
                    }
                gameStats[gameID]["dates"].append(rentalDate)
                gameStats[gameID]["customers"].add(customerID)

    except FileNotFoundError:
        print(f"Warning! Rental.txt not found.")

    # Collect rating data
    pattern = re.compile(r"\[Condition:\s*(.*?)\s*\|\s*Replay:\s*(\d+)")

    try:
        with open("Game_Feedback.txt","r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                gameID = row["GameID"].strip()
                comment = row["Comments"]
                try:
                    rating = float(row["Rating"])
                except ValueError:
                    continue# Pass if rating isn't there

                if gameID not in gameStats:
                    gameStats[gameID] = {
                        "ratings": [],
                        "dates":[],
                        "customers": set(),
                        "replayability": [],
                        "lastCondition": "Unknown"
                    }

                gameStats[gameID]["ratings"].append(rating)

                # Collect extra data
                match = pattern.search(comment)
                if match:
                    condition = match.group(1).strip()
                    replayScore = int(match.group(2))

                    gameStats[gameID]["replayability"].append(replayScore)
                    gameStats[gameID]["lastCondition"] = condition


    except FileNotFoundError:
        print("Warning! Game_Feedback.txt not found.")

    return gameStats

def checkRecency(stats, threshold):
    if not stats["dates"]:
        return True, "Item has never been rented"

    lastRented = max(stats["dates"])
    monthsInactive = (CURRENT_DATE - lastRented).days / 30
    if monthsInactive > threshold:
        return True, f"Inactive for {monthsInactive:.1f} months"
    return False, ""


def checkBreadth(stats, minCustomers):
    unique_count = len(stats["customers"])
    if unique_count < minCustomers:
        return True, f"Niche Appeal (Only {unique_count} unique users)"
    return False, ""

def checkChurnRisk(stats,minUniqueCustomers):
    """Rule 4: Loyalty/Churn Risk (Customer Favorites)"""
    # Logic: If it has LOW breadth but HIGH rating, do NOT prune.
    # It is a "Niche Favorite". Removing it risks churn.

    if not stats["ratings"]: return "KEEP", "No ratings yet" # Safety check
    averageRating = sum(stats["ratings"]) / len(stats["ratings"])
    uniqueCustomers = len(stats["customers"])

    if uniqueCustomers < minUniqueCustomers and averageRating >= CHURN_RISK_RATING:
        return "PROTECT", f"Niche Favorite (Rating {averageRating:.1f})"
    elif uniqueCustomers < minUniqueCustomers:
        return "PRUNE", "Niche but Low Rated"

    return "KEEP", "Broad Appeal"

def smartScore(stats):
    """Rule 5: Weighted Combination"""
    # Weights: Recency (40%), Rating (40%), Breadth (20%)

    if not stats["dates"]: return "KEEP", "No Rental History"
    if not stats["ratings"]: return "KEEP", "No Ratings to Judge"
    # 1. Recency Score (0 to 1, where 1 is recently rented)
    lastRented = max(stats["dates"])
    dasyUnrented = (CURRENT_DATE - lastRented).days
    recencyScore = max(0, 1 - (dasyUnrented / 365)) # 0 if inactive > 1 year

    # 2. Rating Score (divide by 5 to get a value from 0-1)
    ratingScore = sum(stats["ratings"]) / len(stats["ratings"]) / 5.0

    # 3. Breadth Score (Normalized to max 20 customers)
    breadthScore = min(1, len(stats["customers"]) / 20)

    finalScore = (recencyScore * 0.4) + (ratingScore * 0.4) + (breadthScore * 0.2)

    if finalScore < SMART_SCORE_REQUIREMENT:
        return "PRUNE", f"Low Smart Score ({finalScore:.2f})"
    return "KEEP", ""

def generateReport(strategy="recency", threshold=3):
    data = parseFeedbackData()
    report = []
    # Map the slider value to the specific logic needed
    # (e.g. if Recency, threshold is Months. If Breadth, threshold is Customer Count)

    for gameID, stats in data.items():
        action = "KEEP"
        reason = ""

        if strategy == "recency":
            # Pass the dynamic threshold (months) here
            prune, reason = checkRecency(stats, threshold=threshold)
            if prune: action = "PRUNE"

        elif strategy == "breadth":
            # Pass the dynamic threshold (customer count) here
            prune, reason = checkBreadth(stats, minCustomers=threshold)
            if prune: action = "PRUNE"

        elif strategy == "churn":
            action, reason = checkChurnRisk(stats,minUniqueCustomers=threshold)

        elif strategy == "combo":
            action, reason = smartScore(stats)

        if action != "KEEP":
            # Create a dictionary or tuple to make formatting easier later
            report.append((gameID, action, reason))

    return report

# --- 2. LOGIC HANDLERS ---
def onAlgorithmChange(algorithm):
    """
    Updates the explainer text when the user selects a different algorithm.
    This provides immediate context help to the user.
    Parameters:
        change (string): Identifies what algorithm was selected
    """
    thresholdSlider.disabled = False
    thresholdSlider.style.handle_color = None
    selected = algorithm['new']
    if selected == 'recency':
        explanation = (
            "<i><b>Recency:</b> Highlights games that have had ZERO rentals "
            "in the last N months. Useful for clearing dead stock."
            "<br>The slider adjusts the number of months.</i>"
        )

    elif selected == 'breadth':
        explanation = (
            "<i><b>Breadth:</b> Checks how many UNIQUE customers rented the "
            "game. Identifies niche games with low general appeal.<br>"
            "The slider adjusts the minimum number of unique users.<i>"

        )
    elif selected == 'churn':
        explanation = (
            "<i><b>Breadth:</b> Checks how many UNIQUE customers rented the "
            "game with a high rating. Identifies niche games"
            "with low general appeal.<br>The slider adjusts the number of"
            "unique users</i>"
        )
    elif selected == "combo":
        explanation = (
            "<i><b>Weighted Combination:</b> Uses a balanced score of all "
            "metrics to find the statistically worst performing games.</i>"
        )

        thresholdSlider.style.handle_color = '#FF0000'
        thresholdSlider.Value = 1
        thresholdSlider.disabled = True
    else:
        explanation = "<i>Error at onAlgorithmChange</i>"
    algorithmExplainer.value = explanation

# Link the dropdown changes to the handler function


def onAnalysisClick(b):
    # 1. Get values from UI
    chosenStrategy = pruningAlgorithmDropdown.value
    thresholdValue = thresholdSlider.value

    # 2. Update the "Loading" state (optional but good UX)
    recommendationBox.value = """
    <div style="background-color: #fff3cd; color: #856404; padding: 15px; border: 1px solid #ffeeba;">
        <b> Running Analysis...</b>
    </div>
    """

    # 3. Run the logic
    results = generateReport(strategy=chosenStrategy, threshold=thresholdValue)

    # 4. Format the output into HTML
    if not results:
        html_content = f"""
        <div style="background-color: #d4edda; color: #155724; padding: 15px; border: 1px solid #c3e6cb;">
            <b>All Clear!</b><br>
            No games met the criteria for removal based on '{chosenStrategy}' (Threshold: {thresholdValue}).
        </div>
        """
    else:
        # Build a table for the results
        rows = ""
        for game_id, action, reason in results:
            color = "#f8d7da" if action == "PRUNE" else "#fff3cd"
            # Red for Prune, Yellow for Protect/Warn
            rows += f"""
            <tr style="background-color: {color}; border-bottom: 1px solid #ddd;color: black;">
                <td style="padding: 8px;"><b>{game_id}</b></td>
                <td style="padding: 8px;">{action}</td>
                <td style="padding: 8px;">{reason}</td>
            </tr>
            """

        html_content = f"""
        <div style="height: 150px; overflow-y: auto; border: 1px solid #ccc;">
            <table style="width: 100%; border-collapse: collapse; font-family: sans-serif; font-size: 14px;">
                <thead style="background-color: #333; color: white;">
                    <tr>
                        <th style="padding: 8px; text-align: left;">Game ID</th>
                        <th style="padding: 8px; text-align: left;">Action</th>
                        <th style="padding: 8px; text-align: left;">Reason</th>
                    </tr>
                </thead>
                <tbody>
                    {rows}
                </tbody>
            </table>
        </div>
        """

    # 5. Push HTML to the widget
    recommendationBox.value = html_content



# Menu

In [109]:

from IPython.display import display


totalGames, gamesRented = inventoryCount()
gamesAvailable = totalGames - gamesRented
currentOccupancy = getCurrentOccupancy()
maxCapacity = 50
currDate, currTime, openStatus, statusMsg = getStoreStatus()

homeHeader = widgets.HTML(
    f"""
    <div style="display: flex; justify-content: space-between; align-items: center;">
        <h2>Store Dashboard</h2>
        <div style="text-align: right;">
            <b>{currDate}</b><br>
            Time: {currTime}<br>
            Status: {openStatus}
        </div>
    </div>
    """
)

statusLabel = widgets.HTML(f"<i>{statusMsg}</i>")

# --- WIDGET DEFINITIONS ---

# 1. Welcome Header
homeHeader = widgets.HTML("<h2>🏠 Store Dashboard: Live Status</h2>")

# 2. Occupancy Widget (Progress Bar)
occupancyLabel = widgets.Label(value=f"Session Capacity: {currentOccupancy}/{maxCapacity} booked")

occupancyBar = widgets.IntProgress(
    value = currentOccupancy,
    min = 0,
    max = maxCapacity,
    description = "Capacity:",
    bar_style = "danger" if currentOccupancy >= maxCapacity else ("warning" if currentOccupancy > 40 else "success"),
    style = {"bar_color": "#4169E1"}
)

# 3. Inventory Stats
inventoryStats = widgets.HTML(
    value=f"""
    <div style="border: 1px solid #ccc; padding: 10px; border-radius: 5px;">
        <b>Inventory Overview:</b><br>
        <ul style="list-style-type: none; padding: 0;">
            <li><b>Total Games:</b> {totalGames}</li>
            <li><b>Currently Rented:</b> {gamesRented}</li>
            <li><b>Available In-Store:</b> {gamesAvailable}</li>
        </ul>
    </div>
    """
)

"""
---------------------------------------------------------
"""

# --- 1. SEARCH INPUT WIDGETS ---

gameTypeWidget = widgets.Dropdown(
    options = ["Any","Board Game", "Video Game"],
    value = "Any",
    description = "Game Type:"
)
gameTitleInputBox= widgets.Text(
    value = "",
    placeholder = "Enter Title",
    description = "Title:"
)
gameGenreInputBox = widgets.Text(
    value = "",
    placeholder = "Enter Genre",
    description = "Genre:"
)

# --- 2. ACTION BUTTONS ---
searchButton = widgets.Button(
    description = "Search",
    button_style = "success",
    icon = "check",

)
searchButton.style.button_color = '#4169E1'
searchOutput = widgets.Output(layout={'border': '1px solid #ccc', 'padding': '10px'})
searchButton.on_click(onSearchClick)

# --- 3. OUTPUT AREA ---
resultsLabel = widgets.HTML(
    value="<span style='font-size: 25px; font-weight: bold;'>Results</span>",
    layout = widgets.Layout(width='50%', height='50px')
)




# --- 4. UI ASSEMBLY ---
searchTabUI = widgets.VBox([
    widgets.HBox([gameTypeWidget,gameTitleInputBox,gameGenreInputBox]),
    searchButton,
    resultsLabel,
    widgets.HTML("<hr>"),
    searchOutput
])

"""
-------------------------------------------------------
"""

# --- 1. INSTRUCTIONS & WARNINGS ---
warningLabel = widgets.HTML(
    value=("<span style='font-size: 15px; font-weight: bold;'"
    ">Sessions are limited to 50 people. Subscribers may"
    " bring up to 3 guests</span>"),
    layout = widgets.Layout(width='50%', height='80px')
)

# --- 2. USER & DATE INPUTS ---
customerIDInputBox= widgets.Text(
    value = "",
    placeholder = "",
    description = "Customer ID:"
)
# Using DatePicker ensures the date format is consistent (YYYY-MM-DD),
# preventing parsing errors during the booking validation process.
dateInputBox = widgets.DatePicker(
    description = "Date:"
)

# --- 3. SESSION DETAILS ---
sessionTimeWidget = widgets.Dropdown(
    options = ["2pm-6pm", "6pm-10pm"],
    value = "2pm-6pm",
    description = "Game Type:"
)

guestAmountWidget = widgets.Dropdown(
    options = ["0","1","2","3"],
    value = "0",
    description = "Guests (0-3)"
)

# --- 4. ACTION BUTTON ---
bookSessionButton = widgets.Button(
    description = "Book Session",
    button_style = "success",
    icon = "check",
)
bookSessionButton.style.button_color = '#4169E1'

bookingOutput = widgets.Output(
    layout=widgets.Layout(border='1px solid #ddd', padding='10px', height='100px')
)
bookSessionButton.on_click(onBookClick)


# --- 5. UI ASSEMBLY ---
sessionTabUI = widgets.VBox([
    warningLabel,
    widgets.HBox([customerIDInputBox,dateInputBox]),
    widgets.HBox([sessionTimeWidget,guestAmountWidget]),
    bookSessionButton,
    widgets.HTML("<hr>"),
    bookingOutput

])

"""
-------------------------------------------------------
"""
"""
-------------------------------------------------------
"""

# --- 1. RENTAL SECTION UI ---

# Header for the Rental Section
rentGameLabel = widgets.HTML(
    value="<span style='font-size: 20px; font-weight: bold;'>Rent a Game</span>",
    layout = widgets.Layout(width='50%', height='60px')
)
# Input fields for the transaction
rentCustomerIDInputBox= widgets.Text(
    value = "jvsk",
    placeholder = "",
    description = "Customer ID:"
)

gameIDInputBox= widgets.Text(
    value = "bg07",
    placeholder = "",
    description = "Game ID:"
)
# --- STATUS DASHBOARD ---
# Displays dynamic feedback based on user input

customerStatusBox = widgets.HTML(
    value = """
    <div style=
    "background-color: white;
     border: 1px solid #ccc;
     padding: 10px; height: 100px;
     width: 300px;
     overflow-y: auto;
     color: black">
        <b> Rental Status: </b> Enter Customer ID <br>
        <hr style='margin: 5px 0;''>
        Subscription- <br>
        Currently Renting: -
        </div>
      """
)
rentEligibilityBox = widgets.HTML(
    value='''
    <div style=
     "background-color: #f0f0f0;
      border: 1px solid #ccc;
      padding: 10px;
      height: 100px;
      width: 300px;
      display: flex;
      align-items: center;
      justify-content: center;
      text-align: center;
      color: black">
        <b> RENTAL DECISION</b><br>
        Enter IDs to check eligibility.
    </div>
    ''',
    layout=widgets.Layout(width='48%')
)
checkDetailsButton = widgets.Button(
    description="Check Details",
    button_style='info', # Blue color usually indicates "Information"
    icon='search'
)
checkDetailsButton.on_click(onCheckClick)

# The Rent button starts as disabled to prevent invalid rentals
# and/or empty records. It is only enabled if the user satisfies the
# requirements. This is checked by the function updateEligibilityDisplay
rentGameButton = widgets.Button(
    description = "Confirm Rental",
    button_style = "success",
    icon = "check",
    disabled = True
)
rentGameButton.on_click(onRentClick)
outputLog = widgets.Output()

# --- 3. LAYOUT ASSEMBLY ---
sectionDivider = widgets.HTML("<hr>")
rentReturnTabUI = widgets.VBox([
    # Rent Section
    rentGameLabel,
    rentCustomerIDInputBox,
    gameIDInputBox,
    checkDetailsButton,
    widgets.HTML("<br>"),# Small spacer
    widgets.HBox([customerStatusBox, rentEligibilityBox]),
    widgets.HTML("<br>"),
    rentGameButton,
    sectionDivider,
    outputLog

])



"""
-------------------------------------------------------
"""
# --- 1. RETURN SECTION UI ---
returnGameLabel = widgets.HTML(
    value="<span style='font-size: 20px; font-weight: bold;'>Return a Game</span>",
    layout = widgets.Layout(width='50%', height='60px')
)
returnGameID = widgets.Text(
    description="Game ID:",
    layout=widgets.Layout(width='45%')
)

# Bypass the rating system in case an item isn't used
noFeedbackCheckbox = widgets.Checkbox(
    value=False,
    description='Item unused / No rating',
    indent=False
)

# Feedback Input widgets

generalRatingSlider = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    description="Rating",
    layout=widgets.Layout(width='45%'))

replayRatingSlider = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    description="Replayability:",
    layout=widgets.Layout(width='45%'))

conditionDropdown = widgets.Dropdown(
    options=['Perfect', 'Good', 'Worn', 'Damaged'],
    value='Good',
    description='Condition:',
    style={'description_width': 'initial'}
)

returnComment = widgets.Textarea(
    description="Comments:",
    placeholder="Feedback...",
    layout=widgets.Layout(width='95%', height='60px'))

returnButton = widgets.Button(
    description="Accept Return",
    button_style="warning"
)
feedbackSection = widgets.VBox([
    widgets.HTML("<i>Rate the game to help us improve our stock!</i>"),
    generalRatingSlider,
    replayRatingSlider,
    conditionDropdown,
    returnComment
])

returnOutput = widgets.Output()
noFeedbackCheckbox.observe(toggleFeedback, names='value')
returnButton.on_click(on_return_click)

# --- 4. LAYOUT ASSEMBLY ---
returnTabUI = widgets.VBox([
    returnGameLabel,
    returnGameID,
    widgets.HTML("<hr>"),

    noFeedbackCheckbox,
    feedbackSection, # Contains the sliders and comment box

    widgets.HTML("<br>"),
    returnButton,
    returnOutput     # Shows the success/error messages
])


"""
-------------------------------------------------------
"""


# --- 1. CONFIGURATION CONTROLS ---
pruningHeader = widgets.HTML(
    value="<span style='font-size: 25px; font-weight: bold;'>Admin/Pruning</span>",
    layout = widgets.Layout(width='50%', height='60px')
)
# Dropdown allows the user to switch between different pruning logic.
# Providing multiple algorithms (Recency, Appeal etc.) to let
# The user make an informed decision on what to prune
pruningAlgorithmDropdown = widgets.Dropdown(
    options = [
        ("Recency Based (Inactive for N months)", "recency"),
        ('Breadth of Appeal (Unique Customers)', 'breadth'),
        ('Loyalty/Churn Risk (Customer favorites)', 'churn'),
        ('Weighted Combination (Smart Mix)', 'combo')
    ],
    value = "recency",
    style = {"description_width":"initial"},
    layout = widgets.Layout(width="400px")
)

# Dynamic text area that updates to explain the currently selected algorithm.
# This improves usability by documenting the complex logic directly in the UI.
algorithmExplainer = widgets.HTML(
    value = ("<i><b>Recency:</b> Highlights games that have had 0 rentals in"
     "the past N months. Useful for clearing dead stock.<br>"
     "The slider adjusts the number of months.</i>"),
    layout = widgets.Layout(margin="5px 0 10px 0", width="100%")
)

# Slider sets the variable 'N' for the algorithms (e.g., N months or N customers).
thresholdSlider = widgets.IntSlider(
    value=3,
    min=1,
    max=12,
    step=1,
    description='Threshold Value:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

analysisButton = widgets.Button(
    description="Run Analysis",
    button_style="danger",
    icon="bar-chart",
    layout=widgets.Layout(width='200px')
)

recommendationBox = widgets.HTML(
    value="""
    <div style="background-color: #fafafa; border: 1px solid #ccc; padding: 15px; height: 150px; overflow-y: auto;">
        <b>Pruning Recommendations:</b><br>
        <i>Select a strategy and click 'Run Analysis' to see suggestions...</i>
    </div>
    """
)
pruningAlgorithmDropdown.observe(onAlgorithmChange,names="value")

# --- 3. OUTPUTS ---
plotOutput = widgets.Output(
    layout=widgets.Layout(height='350px', border='1px solid #ddd', margin='10px 0')
)

# Scrollable HTML box to display the text-based list of games to remove.
recommendationBox = widgets.HTML(
    value="""
    <div style="
      background-color: #fafafa;
      color: black;
      border: 1px solid #ccc;
      padding: 15px;
      height: 150px;
      overflow-y: auto;">
        <b>Pruning Recommendations:</b><br>
        <i>Select a strategy and click 'Run Analysis' to see suggestions...</i>
    </div>
    """
)
# Link the function to the button
analysisButton.on_click(onAnalysisClick)

adminUI = widgets.VBox([
    pruningHeader,
    pruningAlgorithmDropdown,
    algorithmExplainer,
    thresholdSlider,
    analysisButton,

    sectionDivider,
    recommendationBox

])

"""
-------------------------------------------------------
"""

# --- LAYOUT CONTAINER ---
# This variable 'homeTabContent' will be added to the tabs list
homeTabContent = widgets.VBox([
    homeHeader,
    widgets.HTML("<hr>"),
    occupancyLabel,
    occupancyBar,
    widgets.HTML("<hr>"),
    inventoryStats,
    widgets.HTML("<br>")
])
display(homeTabContent)

# End

In [110]:
# --- MAIN APP ASSEMBLY ---
# 1. Define the list of tabs (The VBox layouts you created previously)
# Note: rentReturnTabUI contains the Rental logic, returnTabUI contains Return logic


children = [
    homeTabContent,    # Tab 0
    searchTabUI,       # Tab 1
    rentReturnTabUI,   # Tab 2
    returnTabUI,       # Tab 3
    sessionTabUI,      # Tab 4
    adminUI            # Tab 5
]

# 2. Initialize the Tab Widget
app = widgets.Tab()
app = widgets.Tab(children=children, layout=widgets.Layout(width='100%', height='480px'))
app.children = children

# 3. Set the Titles for each tab
app.set_title(0, ' Home')
app.set_title(1, ' Search')
app.set_title(2, ' Rent Game')
app.set_title(3, ' Return Game')
app.set_title(4, ' Book Session')
app.set_title(5, ' Admin')

# 4. Display the Final App
display(app)